# Commonality in Liquidity on the National Stock Exchange (NSE) of India (2015–2024)
## Replication, Econometric Modernization, and Empirical Extension of Chordia, Roll & Subrahmanyam (2000, JFE)

---

### Academic & Research Context
This research study conducts an empirical investigation into **commonality in liquidity** across Indian equities listed on the **National Stock Exchange (NSE)** over the 10-year period from **January 2015 to December 2024**. 

We replicate, modernize, and extend the seminal market-microstructure framework established by:
* **Chordia, Roll & Subrahmanyam (2000, *Journal of Financial Economics*)** — *"Commonality in Liquidity"*: Core time-series market model regressions, leave-one-out market and sector liquidity indices, lead/lag adjustment dynamics, and individual volatility shock controls.
* **Karolyi, Lee & van Dijk (2012, *Journal of Financial Economics*)** — *"Understanding Commonality in Liquidity Around the World"*: Global emerging-market priors, demand-side (correlated investor trading) vs. supply-side (funding constraints) liquidity drivers, and systemic crisis amplification.
* **Corwin & Schultz (2012, *Journal of Finance*)** — *"A Simple Way to Estimate Bid-Ask Spreads from Daily High and Low Prices"*: High-low spread proxy separating intraday transaction costs from fundamental volatility using two-day high/low price ratios.
* **Fong, Holden & Trzcinka (2017, *Review of Finance*)** — *"What Are the Best Liquidity Proxies for Global Research?"*: Global proxy evaluations identifying Amihud ILLIQ as the best daily cost-per-volume proxy and Corwin-Schultz (CS) / FHT as optimal spread proxies for low-frequency data.
* **Amihud (2002, *Journal of Financial Markets*)**: Daily illiquidity price impact ratio ($|R| / \text{DVOL} \times 10^6$).
* **Roll (1984, *Journal of Finance*)**: Effective spread from serial autocovariance of daily price returns.
* **Lesmond, Ogden & Trzcinka (1999, *RFS*) / Lesmond (2005, *JFE*)**: Censored-return LOT maximum likelihood transaction cost proxy.

---

### Research Questions Addressed
1. **Baseline Commonality**: Does individual stock liquidity co-move with market-wide and sector-wide liquidity on the NSE after controlling for market returns and firm volatility? *(CRS 2000 replication)*
2. **Market-Cap Cross-Section**: Is liquidity commonality systematically stronger for large-cap equities (Nifty 100) compared to mid-cap (Nifty Midcap 150) and small-cap (Nifty Smallcap 250) firms?
3. **Temporal Structural Shifts**: Did commonality intensify post-2020 following the surge in retail participation (Zerodha/Groww) and algorithmic trading?
4. **Crisis Stress Amplification**: Did commonality experience acute amplification during the COVID-19 March 2020 liquidity dry-up?
5. **Methodological Invariance**: Do findings hold under value-weighted indices, weekly temporal aggregation (mitigating zero-return mass points), and alternative liquidity proxies?

---

### Colab Quick-Start & Execution Workflow
1. **Upload**: Directly upload the project folder `Chordia/` into your Google Drive (e.g. `MyDrive/Chordia/`).
2. **Mount**: Run Cell 1 below. The notebook automatically mounts Google Drive, locates the `Chordia` project directory, sets the working path, and configures all relative paths.
3. **Execute**: Run all cells sequentially (`Runtime` -> `Run all`). The pipeline automatically loads precomputed parquet artifacts (instantaneous) or can re-run all econometric estimations from scratch by setting `FORCE_RERUN = True`.


In [ ]:
# ==============================================================================
# CELL 1: ENVIRONMENT SETUP, GOOGLE DRIVE MOUNTING & PATH RESOLUTION
# ==============================================================================
import os
import sys
import time
import warnings
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown, HTML, Image

warnings.filterwarnings("ignore")

# 1. Environment Detection & Automatic Drive Mounting
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    print("[Colab Setup] Running in Google Colab environment.")
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print(f"  Drive mount notice: {e}")
    
    # Candidate search paths in Google Drive
    candidate_paths = [
        "/content/drive/MyDrive/Chordia",
        "/content/drive/MyDrive/Chordia_Liquidity",
        "/content/drive/MyDrive/My Projects/Chordia",
        "/content/drive/MyDrive/9th_sem/My Projects/Chordia",
        "/content/drive/MyDrive",
        "/content/Chordia",
        "/content"
    ]
    
    project_dir = None
    for path in candidate_paths:
        if os.path.exists(path):
            has_data = os.path.exists(os.path.join(path, "data"))
            has_scripts = os.path.exists(os.path.join(path, "phase4_core_regression.py")) or os.path.exists(os.path.join(path, "phase1_data_cleaning.py"))
            if has_data or has_scripts:
                project_dir = path
                break
                
    if project_dir is None:
        project_dir = "/content/drive/MyDrive/Chordia"
        os.makedirs(project_dir, exist_ok=True)
        
    os.chdir(project_dir)
    if project_dir not in sys.path:
        sys.path.insert(0, project_dir)
    print(f"[Colab Setup] Working directory set to: {os.getcwd()}")
else:
    # Local Machine Environment
    project_dir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    os.chdir(project_dir)
    if project_dir not in sys.path:
        sys.path.insert(0, project_dir)
    print(f"[Local Setup] Working directory: {os.getcwd()}")

# 2. Directory Pointers
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
TABLES_DIR = os.path.join(RESULTS_DIR, "tables")
FIGURES_DIR = os.path.join(RESULTS_DIR, "figures")

for d in [DATA_DIR, RESULTS_DIR, TABLES_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

# 3. Safe Table CSV Loader Helper
def load_table_csv(fname):
    """Safely load result CSV table stripping comments."""
    fpath = os.path.join(TABLES_DIR, fname)
    if not os.path.exists(fpath):
        return None
    try:
        return pd.read_csv(fpath, comment="#")
    except Exception:
        return pd.read_csv(fpath)

# 4. Global Visualization Configuration
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "axes.grid": True,
    "grid.alpha": 0.3,
})
sns.set_theme(style="whitegrid", font="serif")

# Set to True to re-run all estimation pipelines from scratch;
# Set to False to instantly load precomputed parquet panels and results tables.
FORCE_RERUN = False

print("-" * 75)
print("PROJECT CONFIGURATION SUMMARY")
print(f"  Working Base Directory : {BASE_DIR}")
print(f"  Data Directory         : {DATA_DIR}")
print(f"  Results Directory      : {RESULTS_DIR}")
print(f"  Tables Directory       : {TABLES_DIR}")
print(f"  Figures Directory      : {FIGURES_DIR}")
print(f"  FORCE_RERUN Flag       : {FORCE_RERUN}")
print("-" * 75)

## Phase 1 — Data Acquisition, Cleaning & Dynamic Nifty 500 Universe Construction

### Econometric & Microstructure Foundation
Accurate empirical market microstructure research requires meticulous data hygiene to eliminate sample-selection bias, survivorship bias, and spurious outliers:

1. **Survivorship-Bias-Free Dynamic Index Universe**:
   Constituency lists that only reflect index membership at a single static point in time introduce severe survivorship bias. We resolve this by parsing `osc_sheet_company_Included_On.csv`, which tracks **96 historical index rebalancing snapshots** across 2015–2024. For each trading day $t$, a stock is included if and only if it was an active constituent in the most recent preceding snapshot date $S \le t$.
2. **Contiguous Trading Spell Segmentation (`spell_id`)**:
   When firms temporarily exit and later re-enter the Nifty 500 (e.g. after a multi-year hiatus), computing lagged returns across calendar gaps yields catastrophic artificial returns. We segment every stock's timeline into contiguous trading spells where calendar gaps exceeding 10 days define a new spell boundary (`spell_id`). Multi-year gap returns on re-entry dates are strictly set to `NaN`.
3. **Data Quality & Tradability Filters**:
   - **Penny Stock Screen**: Exclude observations with adjusted closing price $P_{i,t} < ₹5$ to avoid tick-size distortion and discrete microstructure noise.
   - **Zero-Volume Filter**: Drop non-trading days where rupee volume $\text{DVOL}_{i,t} = 0$.
   - **Thin-Trading Screen**: Drop stock-years with $>20\%$ missing observations across the calendar year.
4. **Logarithmic Return Formulation**:
   $$R_{i,t} = \ln\left(\frac{P_{i,t}^{\text{adj}}}{P_{i,t-1}^{\text{adj}}}\right)$$
5. **Annual Winsorization**:
   All continuous liquidity measures and variables are winsorized at the 1st and 99th percentiles **within each calendar year** to handle extreme tails without distorting secular multi-year trends.


In [ ]:
# ==============================================================================
# CELL 2: PHASE 1 -- LOAD & VERIFY CLEANED PANEL DATA
# ==============================================================================
clean_panel_path = os.path.join(DATA_DIR, "panel_clean.parquet")

if not os.path.exists(clean_panel_path) or FORCE_RERUN:
    print("[Phase 1] Executing phase1_data_cleaning.py script...")
    %run phase1_data_cleaning.py
else:
    print(f"[Phase 1] Cleaned panel found: {clean_panel_path}")

# Load cleaned panel and display structural diagnostics
df_clean = pd.read_parquet(clean_panel_path)
df_clean["Date"] = pd.to_datetime(df_clean["Date"])

print("=" * 75)
print("PHASE 1 CLEANED DATASET DIAGNOSTICS")
print("=" * 75)
print(f"  Total Cleaned Observations : {df_clean.shape[0]:,}")
print(f"  Total Cleaned Columns      : {df_clean.shape[1]}")
print(f"  Unique Constituents        : {df_clean['Company Name'].nunique():,}")
print(f"  Temporal Span              : {df_clean['Date'].min().strftime('%Y-%m-%d')} to {df_clean['Date'].max().strftime('%Y-%m-%d')}")
print(f"  Total Trading Days         : {df_clean['Date'].nunique():,}")
print(f"  Average Stocks per Day     : {df_clean.groupby('Date')['Company Name'].count().mean():.1f}")
print("=" * 75)

# Annual breakdown of constituent counts
annual_summary = df_clean.groupby("Year").agg(
    Trading_Days=("Date", "nunique"),
    Unique_Stocks=("Company Name", "nunique"),
    Total_Observations=("Company Name", "count"),
    Avg_Daily_DVOL_Cr=("dvol", lambda x: (x.mean() / 1e7)),
    Median_MktCap_Cr=("mktcap", lambda x: (x.median() / 1e7))
).reset_index()

print("\nAnnual Panel Structure (2015 - 2024):")
display(annual_summary)

print("\nSample Records (Cleaned Panel):")
display(df_clean[["Date", "Year", "Company Name", "symbol", "close", "dvol", "mktcap", "log_return"]].head(5))

## Phase 2 — Microstructure Liquidity Measures Construction & Quality Control

### Theoretical Specifications & Proxy Hierarchy
Given daily OHLCV data without intraday limit-order-book quotes, we deploy the empirical proxy hierarchy validated by **Fong, Holden & Trzcinka (2017)**:

1. **Amihud (2002) Illiquidity Ratio (Primary Daily Cost-per-Volume)**:
   $$\text{ILLIQ}_{i,t} = \frac{|R_{i,t}|}{\text{DVOL}_{i,t}} \times 10^6$$
   Measures price impact per unit of rupee volume. Scaled by $10^6$ for interpretability in Indian equity markets.
2. **Corwin & Schultz (2012) High-Low Spread (Primary Daily Spread Proxy)**:
   Exploits the insight that 2-day price ranges reflect both daily volatility and bid-ask spread:
   $$\beta_t = \left[\ln\left(\frac{H_t}{L_t}\right)\right]^2 + \left[\ln\left(\frac{H_{t-1}}{L_{t-1}}\right)\right]^2$$
   $$\gamma_t = \left[\ln\left(\frac{\max(H_t, H_{t-1})}{\min(L_t, L_{t-1})}\right)\right]^2$$
   $$\alpha_t = \frac{\sqrt{2\beta_t} - \sqrt{\beta_t}}{3 - 2\sqrt{2}} - \sqrt{\frac{\gamma_t}{3 - 2\sqrt{2}}}$$
   $$CS_{i,t} = \frac{2(e^{\alpha_t} - 1)}{1 + e^{\alpha_t}}, \quad \text{floored at } 0 \text{ when } \alpha_t < 0$$
   *Note*: Negative spread estimates occurring from overnight price drift are set to 0 per Corwin & Schultz (2012).
3. **Roll (1984) Effective Spread (Secondary Daily Proxy)**:
   $$ROLL_{i,t} = 2\sqrt{\max\left(-\widehat{\text{Cov}}(\Delta p_t, \Delta p_{t-1}), 0\right)}$$
   Estimated over a rolling 21-day trading window within contiguous spells.
4. **Fong, Holden & Trzcinka (2017) FHT Spread (Primary Monthly Spread Proxy)**:
   $$FHT_{i,m} = 2\sigma_{i,m} \Phi^{-1}\left(\frac{1 + z_{i,m}}{2}\right)$$
   Where $\sigma_{i,m}$ is the standard deviation of non-zero daily returns, $z_{i,m}$ is the fraction of zero-return days in month $m$, and $\Phi^{-1}$ is the inverse standard normal CDF.
5. **Amivest Liquidity Ratio (Supervisor Robustness Check)**:
   $$\text{Amivest}_{i,m} = \frac{\sum_{t \in m} \text{DVOL}_{i,t}}{\sum_{t \in m} |R_{i,t}|} \times 10^{-6}$$
   Constructed as the monthly ratio of sums to maintain mathematical consistency with market depth.
6. **Lesmond, Ogden & Trzcinka (1999) / Lesmond (2005) LOT Measure (MLE)**:
   Censored-regression probit modeling of zero returns bounded by effective trading cost thresholds $\alpha_1 < 0 < \alpha_2$.
7. **CRS (2000) Proportional Innovations for Regressions**:
   $$\Delta L_{i,t} = \frac{L_{i,t} - L_{i,t-1}}{(|L_{i,t}| + |L_{i,t-1}|)/2} \in [-2.0, +2.0]$$
   Midpoint percentage growth bounded in $[-200\%, +200\%]$ for non-negative measures.


In [ ]:
# ==============================================================================
# CELL 3: PHASE 2 -- LOAD & VERIFY LIQUIDITY MEASURES & CORRELATIONS
# ==============================================================================
liq_panel_path = os.path.join(DATA_DIR, "panel_with_liquidity.parquet")
monthly_panel_path = os.path.join(DATA_DIR, "monthly_liquidity_panel.parquet")

if not os.path.exists(liq_panel_path) or FORCE_RERUN:
    print("[Phase 2] Executing phase2_liquidity_measures.py...")
    %run phase2_liquidity_measures.py
else:
    print(f"[Phase 2] Liquidity panel found: {liq_panel_path}")

df_liq = pd.read_parquet(liq_panel_path)
df_monthly = pd.read_parquet(monthly_panel_path)

print("=" * 75)
print("PHASE 2 LIQUIDITY MEASURES SUMMARY STATISTICS")
print("=" * 75)

# Primary daily measures summary
measures = ["illiq", "cs_spread", "roll", "d_illiq", "d_cs", "d_roll"]
liq_summary = df_liq[measures].describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]).T
liq_summary = liq_summary[["count", "mean", "std", "5%", "25%", "50%", "75%", "95%"]]
liq_summary.rename(index={
    "illiq": "Amihud (2002) ILLIQ (x 10^6)",
    "cs_spread": "Corwin-Schultz (2012) Spread",
    "roll": "Roll (1984) Effective Spread",
    "d_illiq": "CRS Proportional Change d_illiq",
    "d_cs": "CRS Proportional Change d_cs",
    "d_roll": "CRS Proportional Change d_roll"
}, inplace=True)
display(liq_summary.round(4))

# Verification of expected correlation properties
print("\nCorrelation Matrix Across Daily Liquidity Levels:")
corr_levels = df_liq[["illiq", "cs_spread", "roll"]].corr()
display(corr_levels.round(4))

# Cross-sectional Amivest vs 1/Amihud check (Supervisor sanity check)
firm_monthly = df_monthly.groupby("Company Name")[["amivest_monthly", "illiq_mean"]].mean().dropna()
firm_monthly["inv_amihud"] = 1.0 / firm_monthly["illiq_mean"].replace(0, np.nan)
valid_firm = firm_monthly.dropna()
p_corr, _ = stats.pearsonr(valid_firm["amivest_monthly"], valid_firm["inv_amihud"])
s_corr, _ = stats.spearmanr(valid_firm["amivest_monthly"], valid_firm["inv_amihud"])

print("-" * 75)
print("SANITY CHECK: AMIVEST VS. 1/AMIHUD INVERSE RELATIONSHIP")
print(f"  Firm-Level Pearson Correlation(Amivest, 1/Amihud)  : {p_corr:.4f} (Expected > 0.90)")
print(f"  Firm-Level Spearman Correlation(Amivest, 1/Amihud) : {s_corr:.4f} (Expected > 0.90)")
print(f"  Status: {'PASS [OK]' if p_corr > 0.90 else 'FLAGGED'}")
print("-" * 75)

## Phase 3 — Market & Sector Liquidity Indices & Control Variables

### Econometric Methodology
A central insight from **Chordia, Roll & Subrahmanyam (2000)** is that estimating commonality by regressing a stock's liquidity on the unadjusted market average creates severe **mechanical endogeneity** (self-correlation), because stock $i$ is part of the market sum.

1. **Leave-One-Out ("Exclude-Own") Market Liquidity Index**:
   $$\Delta\overline{L}_{M,-i,t} = \frac{1}{N_t - 1}\sum_{j \neq i} \Delta L_{j,t}$$
   By strictly purging stock $i$'s own innovation from the market index, we ensure that $\hat\beta_{i,M}$ reflects genuine market-wide co-movement rather than self-inclusion bias. Both equal-weighted (EW) and value-weighted (VW) indices are constructed.
2. **Leave-One-Out Sector Liquidity Index**:
   $$\Delta\overline{L}_{S,-i,t} = \frac{1}{N_{s,t} - 1}\sum_{\substack{j \neq i \\ j \in \text{Sector}(i)}} \Delta L_{j,t}$$
   Sectors are mapped using standard 12–14 industry classifications / NIC Sections. To prevent noisy or undefined indices, observations are required to have $N_{s,t} \ge 5$ stocks in the sector on date $t$.
3. **Market Return Control Variables**:
   Liquidity co-moves with broad market price movements. To isolate liquidity-specific commonality from return-driven volume changes, we include **concurrent, lead (+1), and lagged (-1)** market returns:
   $$R_{M,t}, \quad R_{M,t+1}, \quad R_{M,t-1}$$
4. **Individual Firm Volatility Controls**:
   Following CRS (2000, Table 3 footnote), we control for idiosyncratic firm-level volatility shocks using the proportional daily change in squared returns:
   $$\Delta(R_{i,t}^2) = \frac{R_{i,t}^2 - R_{i,t-1}^2}{(R_{i,t}^2 + R_{i,t-1}^2)/2}$$
   As well as rolling 21-day return volatility shocks $\Delta\sigma_{i,t}^{\text{vol21d}}$.


In [ ]:
# ==============================================================================
# CELL 4: PHASE 3 -- MARKET & SECTOR INDICES & CONTROL VARIABLES
# ==============================================================================
import phase3_market_sector_indices as p3

indices_panel_path = os.path.join(DATA_DIR, "panel_with_indices.parquet")

if not os.path.exists(indices_panel_path) or FORCE_RERUN:
    print("[Phase 3] Generating market & sector indices via phase3_market_sector_indices.run_phase3()...")
    df_indices = p3.run_phase3()
else:
    print(f"[Phase 3] Enriched panel found: {indices_panel_path}")
    df_indices = pd.read_parquet(indices_panel_path)

print("=" * 75)
print("PHASE 3 ENRICHED PANEL OVERVIEW")
print("=" * 75)
print(f"  Rows: {df_indices.shape[0]:,} | Columns: {df_indices.shape[1]}")
print(f"  Unique Stocks: {df_indices['Company Name'].nunique():,}")

# Verify leave-one-out columns
index_cols = [c for c in df_indices.columns if "mkt_ew" in c or "sector_ew" in c or "ret_mkt" in c or "ret_sq" in c]
print(f"  Constructed Index & Control Columns ({len(index_cols)}):")
for col in index_cols[:12]:
    print(f"    - {col}")
print("    ... [additional lag/lead terms present]")

# Display time-series sample of indices
df_indices["Date"] = pd.to_datetime(df_indices["Date"])
sample_indices = df_indices[df_indices["symbol"] == "RELIANCE"][
    ["Date", "symbol", "d_illiq", "d_illiq_mkt_ew", "d_illiq_sector_ew", "ret_mkt_ew", "d_ret_sq"]
].dropna().head(5)
print("\nSample Leave-One-Out Indices for RELIANCE:")
display(sample_indices)

## Phase 4 — Core CRS (2000) Commonality-in-Liquidity Regressions

### Econometric Specification
We estimate stock-by-stock time-series Ordinary Least Squares (OLS) regressions with **Newey-West (1987) Heteroscedasticity and Autocorrelation Consistent (HAC)** standard errors (5 lags, Bartlett kernel), strictly adhering to **Chordia, Roll & Subrahmanyam (2000, Table 3)**:

$$\Delta L_{i,t} = \alpha_i + \beta_{i,M}^0 \Delta\overline{L}_{M,-i,t} + \beta_{i,M}^+ \Delta\overline{L}_{M,-i,t+1} + \beta_{i,M}^- \Delta\overline{L}_{M,-i,t-1} + \sum_{\tau=-1}^{+1} \gamma_{i,\tau} R_{M,t+\tau} + \delta_i \Delta\sigma_{i,t} + \varepsilon_{i,t}$$

Where:
* $\beta_{i,M}^0$: **Contemporaneous Market Commonality Beta** — primary metric of systematic liquidity co-movement.
* $\beta_{i,M}^+, \beta_{i,M}^-$: **Lead (+1) and Lag (-1) Betas** capturing non-synchronous trading adjustments.
* $\text{Sum }\beta_i = \beta_{i,M}^0 + \beta_{i,M}^+ + \beta_{i,M}^-$: **Cumulative Systematic Market Commonality**.
* $R_{M,t+\tau}$: Concurrent, lead, and lag market returns controlling for market returns.
* $\Delta\sigma_{i,t} = \Delta(R_{i,t}^2)$: Proportional idiosyncratic volatility shocks.

### Summary Table 2 Structure:
* **Panel A**: Equal-Weighted Market Model across all three primary proxies (Amihud, Corwin-Schultz, Roll).
* **Panel B**: Market + Sector Model isolating incremental sector commonality ($\text{Sum }\beta_{i,S}$).
* **Panel C**: Size Strata Breakdown across **Nifty 100** (Large-cap), **Nifty Midcap 150** (Mid-cap), and **Nifty Smallcap 250** (Small-cap).
* **Panel D**: Size Quintiles verifying monotonic commonality gradients across market capitalizations.


In [ ]:
# ==============================================================================
# CELL 5: PHASE 4 -- CORE CRS (2000) REGRESSIONS & TABLE 2 RESULTS
# ==============================================================================
import phase4_core_regression as p4

stock_betas_file = os.path.join(DATA_DIR, "stock_betas_phase4.parquet")

if not os.path.exists(stock_betas_file) or FORCE_RERUN:
    print("[Phase 4] Estimating stock-by-stock regressions via phase4_core_regression.run_phase4()...")
    stock_betas_df, p4_panels = p4.run_phase4()
else:
    print(f"[Phase 4] Precomputed stock betas found: {stock_betas_file}")
    stock_betas_df = pd.read_parquet(stock_betas_file)

# Display Table 2 Panels
print("=" * 80)
print("TABLE 2: PANEL A -- EQUAL-WEIGHTED MARKET MODEL")
print("=" * 80)
t2a = load_table_csv("table2_panelA_market_model.csv")
if t2a is not None:
    display(t2a)

print("\n" + "=" * 80)
print("TABLE 2: PANEL B -- MARKET + SECTOR COMMONALITY MODEL")
print("=" * 80)
t2b = load_table_csv("table2_panelB_sector_model.csv")
if t2b is not None:
    display(t2b)

print("\n" + "=" * 80)
print("TABLE 2: PANEL C -- SIZE STRATA BREAKDOWN (NIFTY 100 vs. MIDCAP 150 vs. SMALLCAP 250)")
print("=" * 80)
t2c = load_table_csv("table2_size_breakdown.csv")
if t2c is not None:
    display(t2c.head(14))

print("\n" + "=" * 80)
print("TABLE 2: PANEL D -- SIZE QUINTILES (Q1 = LARGEST TO Q5 = SMALLEST)")
print("=" * 80)
t2d = load_table_csv("table2_size_quintiles.csv")
if t2d is not None:
    display(t2d.head(10))

## Phase 5 — Cross-Sectional Analysis & Firm Determinants (Size & Liquidity Gradients)

### Econometric Specification & Dependent Variable Properties
Having estimated stock-level commonality betas $\hat\beta_i$ in Phase 4, we analyze which firm characteristics explain cross-sectional variation in systematic liquidity risk:

$$\hat\beta_i = a_0 + a_1 \ln(\overline{\text{MktCap}}_i) + a_2 \ln(\overline{\text{Price}}_i) + a_3 \overline{\text{Turnover}}_i + a_4 \overline{\sigma}_i + \varepsilon_i$$

### Critical Econometric Consideration:
* **Why Shanken (1992) does NOT apply**: The Shanken correction is designed for two-pass asset pricing tests where estimated betas appear on the **right-hand side (RHS)** as explanatory regressors to explain stock returns. Here, the estimated commonality beta $\hat\beta_i$ is the **dependent variable (LHS)**.
* **Econometrically Correct Estimation Methods**:
  1. **Ordinary Least Squares (OLS) with HC3 / White Robust Standard Errors**: Standard baseline accounting for heteroscedasticity.
  2. **Precision-Weighted Least Squares (WLS)**: We weight each firm by the inverse standard error of its first-stage beta estimate ($w_i = 1 / \widehat{\text{SE}}(\hat\beta_i)$), downweighting noisily estimated betas.
  3. **Annual Fama & MacBeth (1973) Regressions**: Cross-sectional regressions estimated year-by-year across all 10 years (2015–2024), aggregating coefficients via Newey-West standard errors across time.


In [ ]:
# ==============================================================================
# CELL 6: PHASE 5 -- CROSS-SECTIONAL DETERMINANTS & TABLE 3
# ==============================================================================
import phase5_cross_sectional_analysis as p5

t3_file = os.path.join(TABLES_DIR, "table3_cross_sectional_determinants.csv")

if not os.path.exists(t3_file) or FORCE_RERUN:
    print("[Phase 5] Estimating cross-sectional models via phase5_cross_sectional_analysis.run_phase5()...")
    t3_master, t3_sub, cross_df, annual_df = p5.run_phase5()
else:
    print(f"[Phase 5] Precomputed Table 3 found: {t3_file}")

# Display Table 3 Panels
print("=" * 80)
print("TABLE 3: PANEL A -- ORDINARY LEAST SQUARES (OLS HC3)")
print("=" * 80)
t3a = load_table_csv("table3_panelA_ols.csv")
if t3a is not None:
    display(t3a)

print("\n" + "=" * 80)
print("TABLE 3: PANEL B -- PRECISION-WEIGHTED LEAST SQUARES (WLS)")
print("=" * 80)
t3b = load_table_csv("table3_panelB_wls.csv")
if t3b is not None:
    display(t3b)

print("\n" + "=" * 80)
print("TABLE 3: PANEL C -- ANNUAL FAMA-MACBETH REGRESSIONS")
print("=" * 80)
t3c = load_table_csv("table3_panelC_fama_macbeth.csv")
if t3c is not None:
    display(t3c)

print("\n" + "=" * 80)
print("PAIRWISE SIZE STRATA HYPOTHESIS TESTS (LARGE vs. MID vs. SMALL)")
print("=" * 80)
t3_size = load_table_csv("table3_size_pairwise_tests.csv")
if t3_size is not None:
    display(t3_size)

## Phase 6 — Temporal Sub-Period Analysis & Macro Determinants

### Research Context & Structural Break Hypotheses
A central thesis inquiry is whether the **post-2020 retail investing explosion** (via discount brokerages such as Zerodha and Groww) and surging algorithmic participation structurally altered liquidity commonality on the NSE.

1. **Sub-Period Sample Partitioning**:
   * **Pre-COVID Baseline**: 2016–2019 (4 full calendar years before pandemic).
   * **Post-COVID Era**: 2021–2024 (4 full calendar years with surging retail participation).
   * *Excluded Transition Year*: 2020 (acute shock year isolated and analyzed in Phase 7).
2. **Paired vs. Unpaired Hypothesis Testing**:
   Because the same firms exist across both regimes, standard two-sample independent tests (e.g. Welch's t-test) are invalid. We employ **paired t-tests**, non-parametric **Wilcoxon signed-rank tests**, and **sign tests**.
3. **Correcting for Common Date Correlation (Petersen 2009)**:
   Because all stocks share identical calendar dates, estimation errors across firms are cross-sectionally correlated. We address this using:
   * **Date-Clustered Pooled Panel Regressions** with firm fixed effects.
   * **Calendar-Day Block Bootstrap** (2,000 resamples across trading days) per Politis & Romano (1994).
4. **Macroeconomic Determinants of Market-Wide Liquidity Innovations (CRS 2000 Table 5)**:
   Regressing daily market liquidity innovations $\Delta\overline{L}_{M,t}$ on contemporaneous market returns ($R_{M,t}^+, R_{M,t}^-$), return volatility shocks $\Delta(R_{M,t}^2)$, day-of-week seasonality (Monday/Friday), and regime indicators.


In [ ]:
# ==============================================================================
# CELL 7: PHASE 6 -- TEMPORAL SUB-PERIOD ANALYSIS & TABLE 4
# ==============================================================================
import phase6_subperiod_analysis as p6

t4_file = os.path.join(TABLES_DIR, "table4_subperiod_comparison.csv")

if not os.path.exists(t4_file) or FORCE_RERUN:
    print("[Phase 6] Estimating sub-period shifts via phase6_subperiod_analysis.run_phase6()...")
    sub_betas, t4_df, roll_df, macro_res, p6_artifacts = p6.run_phase6()
else:
    print(f"[Phase 6] Precomputed Table 4 found: {t4_file}")

print("=" * 80)
print("TABLE 4: PANEL A -- SUB-PERIOD COMMONALITY MEANS (PRE-COVID vs. POST-COVID)")
print("=" * 80)
t4a = load_table_csv("table4_panelA_subperiod_means.csv")
if t4a is not None:
    display(t4a)

print("\n" + "=" * 80)
print("TABLE 4: PANEL B -- PAIRED STATISTICAL HYPOTHESIS TESTS")
print("=" * 80)
t4b = load_table_csv("table4_panelB_paired_tests.csv")
if t4b is not None:
    display(t4b)

print("\n" + "=" * 80)
print("TABLE 4: PANEL D -- MACROECONOMIC & MARKET-STATE DETERMINANTS (CRS 2000 TABLE 5)")
print("=" * 80)
t4d = load_table_csv("table4_market_state_determinants.csv")
if t4d is not None:
    display(t4d)

## Phase 7 — COVID-19 Acute Stress Crisis Interaction Analysis

### The March 2020 Dual Shock
During the acute onset of the pandemic (**February 15 to April 30, 2020**), global markets experienced severe liquidity dry-ups. In India, this coincided with the **Yes Bank moratorium (March 5, 2020)**, creating a concentrated dual liquidity and solvency shock.

### Econometric Interaction Specification:
We evaluate whether systematic commonality intensified during the acute crisis window using an explicit interaction model:

$$\Delta L_{i,t} = \alpha_i + \beta_{i,M}^0 \Delta\overline{L}_{M,-i,t} + \phi_{i,\text{Crisis}} \left(\Delta\overline{L}_{M,-i,t} \times \mathbf{1}_{\text{Crisis},t}\right) + \theta_{i,\text{Crisis}} \mathbf{1}_{\text{Crisis},t} + \sum_{\tau=-1}^{+1} \gamma_{i,\tau} R_{M,t+\tau} + \delta_i \Delta\sigma_{i,t} + \varepsilon_{i,t}$$

Where:
* $\phi_{i,\text{Crisis}} > 0$: **Crisis Amplification Coefficient** — indicates that systematic liquidity co-movement intensified during the liquidity dry-up.
* Standard errors are clustered simultaneously by **stock and date** (two-way clustering per Cameron, Gelbach & Miller 2011).


In [ ]:
# ==============================================================================
# CELL 8: PHASE 7 -- COVID-19 CRISIS INTERACTION ANALYSIS & TABLE 5
# ==============================================================================
import phase7_covid_crisis as p7

t5_file = os.path.join(TABLES_DIR, "table5_covid_crisis_interaction.csv")

if not os.path.exists(t5_file) or FORCE_RERUN:
    print("[Phase 7] Estimating COVID-19 crisis interaction models via run_phase7()...")
    crisis_betas, t5_df, pooled_res, p7_artifacts = p7.run_phase7()
else:
    print(f"[Phase 7] Precomputed Table 5 found: {t5_file}")

print("=" * 80)
print("TABLE 5: PANEL A -- INDIVIDUAL STOCK CRISIS INTERACTION DISTRIBUTION")
print("=" * 80)
t5a = load_table_csv("table5_panelA_stock_distribution.csv")
if t5a is not None:
    display(t5a)

print("\n" + "=" * 80)
print("TABLE 5: PANEL B -- POOLED PANEL REGRESSION WITH TWO-WAY CLUSTERING")
print("=" * 80)
t5b = load_table_csv("table5_panelB_pooled_panel.csv")
if t5b is not None:
    display(t5b)

print("\n" + "=" * 80)
print("TABLE 5: PANEL C -- SIZE STRATA CRISIS SENSITIVITY")
print("=" * 80)
t5c = load_table_csv("table5_panelC_size_strata.csv")
if t5c is not None:
    display(t5c)

## Phase 8 — Comprehensive Methodological Robustness Checks

### Multidimensional Sensitivity Analyses (Table 6)
To ensure empirical findings are not artifacts of specific methodological choices, we test robustness across four distinct dimensions:

* **Panel A: Value-Weighted Market & Sector Commonality**:
  Re-estimating the core market model with value-weighted leave-one-out market and sector indices (weights based on market capitalization).
* **Panel B: Weekly Temporal Aggregation & Mass Point Mitigation**:
  A known property of daily high-low spread estimators (Corwin & Schultz 2012) in active equity markets is that zero-return days and overnight jumps produce discrete mass points at 0 and $\pm 2$. Aggregating to weekly frequency mitigates discrete daily noise.
* **Panel C: Individual Firm Volatility Controls Sensitivity**:
  Evaluating commonality estimates with alternative volatility controls (rolling 21-day realized volatility vs. squared return innovations vs. unadjusted models).
* **Panel D: Subsamples, Portfolios & Alternative Proxies**:
  Testing large-cap Nifty 100 constituents alone, size-sorted decile portfolios, and alternative liquidity proxies including **Roll (1984)**, **Fong, Holden & Trzcinka (2017) FHT spread**, and the **Amivest liquidity ratio**.


In [ ]:
# ==============================================================================
# CELL 9: PHASE 8 -- METHODOLOGICAL ROBUSTNESS CHECKS & TABLE 6
# ==============================================================================
import phase8_robustness_checks as p8

t6_file = os.path.join(TABLES_DIR, "table6_robustness_checks.csv")

if not os.path.exists(t6_file) or FORCE_RERUN:
    print("[Phase 8] Executing robustness checks via phase8_robustness_checks.run_phase8()...")
    rob_betas, rob_panels, rob_artifacts = p8.run_phase8()
else:
    print(f"[Phase 8] Precomputed Table 6 found: {t6_file}")

# Display Table 6 Panels
for panel_name, panel_file in [
    ("PANEL A: VALUE-WEIGHTED SPECIFICATIONS", "table6_panelA_value_weighted.csv"),
    ("PANEL B: WEEKLY AGGREGATION & MASS POINT MITIGATION", "table6_panelB_weekly_spread.csv"),
    ("PANEL C: VOLATILITY CONTROLS SENSITIVITY", "table6_panelC_volatility_controls.csv"),
    ("PANEL D: SUBSAMPLES, PORTFOLIOS & ALTERNATIVE PROXIES", "table6_panelD_subsamples_portfolios.csv")
]:
    t6_panel = load_table_csv(panel_file)
    if t6_panel is not None:
        print("\n" + "=" * 80)
        print(f"TABLE 6: {panel_name}")
        print("=" * 80)
        display(t6_panel)

## Phase 9 — Publication Figures, Summary Statistics, & Empirical Benchmarking

### Publication Deliverables:
1. **Table 1: Comprehensive Summary Statistics Panel**:
   * **Panel A**: Full Sample Summary Statistics across all liquidity measures and innovations.
   * **Panel B**: Size Strata Breakdown (Nifty 100 vs. Midcap 150 vs. Smallcap 250).
2. **High-Resolution Publication Visualizations (Figures 1 to 5)**:
   * **Figure 1**: 10-Year Time Series of Market-Wide Liquidity (Amihud & CS Spreads) with Macro Stress Episodes.
   * **Figure 2**: Cross-Sectional Size Gradient (Commonality Beta across Market-Cap Quintiles).
   * **Figure 3**: Rolling 252-Day Commonality Beta Evolution (2016–2024).
   * **Figure 4**: Size Quintile Distributional Boxplots.
   * **Figure 5**: COVID-19 Acute Crisis Zoom & Spread Decomposition.
3. **Empirical Sanity Checks against Literature Benchmarks**:
   * Monotonicity of Amihud ILLIQ across market caps.
   * Monotonicity of Corwin-Schultz spreads across market caps.
   * Amivest vs. Amihud inverse correlation check ($> 0.90$).
   * Comparison against US benchmarks (CRS 2000) and emerging market priors (Karolyi et al. 2012).


In [ ]:
# ==============================================================================
# CELL 10: PHASE 9 -- PUBLICATION ASSETS, FIGURES & SANITY CHECKS
# ==============================================================================
import phase9_figures_and_tables as p9

# Run Phase 9 pipeline to generate/verify all publication assets
print("[Phase 9] Running Phase 9 pipeline...")
p9_results = p9.run_phase9_pipeline(base_dir=BASE_DIR)

# Display Table 1 Panels
print("=" * 80)
print("TABLE 1: PANEL A -- FULL SAMPLE SUMMARY STATISTICS")
print("=" * 80)
t1a = load_table_csv("table1_panelA_full_sample.csv")
if t1a is not None:
    display(t1a)

print("\n" + "=" * 80)
print("TABLE 1: PANEL B -- SUMMARY STATISTICS BY SIZE STRATA")
print("=" * 80)
t1b = load_table_csv("table1_panelB_size_strata.csv")
if t1b is not None:
    display(t1b)

# Display Empirical Sanity Checks JSON Report
sanity_path = os.path.join(RESULTS_DIR, "empirical_sanity_checks.json")
if os.path.exists(sanity_path):
    import json
    with open(sanity_path, "r") as f:
        sanity_data = json.load(f)
    print("\n" + "=" * 80)
    print("EMPIRICAL SANITY CHECKS REPORT")
    print("=" * 80)
    c1 = sanity_data.get("check1_monotonic_size_gradient_amihud", {})
    c2 = sanity_data.get("check2_spread_tightness_large_caps", {})
    c3 = sanity_data.get("check3_amivest_inverse_correlation", {})
    lit = sanity_data.get("literature_benchmarks", {})
    karolyi = lit.get("karolyi_2012_emerging_market_prior", {})
    
    print(f"  1. Size Gradient (Amihud)     : {'PASS [OK]' if c1.get('passed') else 'FAIL'}")
    print(f"     Large ({c1.get('large_cap_mean', 0):.6f}) < Mid ({c1.get('mid_cap_mean', 0):.6f}) < Small ({c1.get('small_cap_mean', 0):.6f}) < Micro ({c1.get('micro_cap_mean', 0):.6f})")
    print(f"  2. Size Gradient (CS Spread)  : {'PASS [OK]' if c2.get('passed') else 'FAIL'}")
    print(f"     Large ({c2.get('large_cap_mean', 0):.5f}) < Mid ({c2.get('mid_cap_mean', 0):.5f}) < Small ({c2.get('small_cap_mean', 0):.5f}) < Micro ({c2.get('micro_cap_mean', 0):.5f})")
    print(f"  3. Amivest Ratio Verification : {'PASS [OK]' if c3.get('passed') else 'FAIL'}")
    print(f"     Pooled Pearson: {c3.get('pearson_pooled', 0):.4f}, Firm-Level Pearson: {c3.get('pearson_firm_level', 0):.4f} (> 0.90)")
    print(f"  4. Emerging Market Prior      : {'SUPPORTED [OK]' if karolyi.get('empirically_supported') else 'UNSUPPORTED'}")
    print(f"     {karolyi.get('evidence', '')}")

# Inline Display of Figures 1 to 5
print("\n" + "=" * 80)
print("PUBLICATION FIGURES (1 THROUGH 5)")
print("=" * 80)

figures_to_display = [
    ("Figure 1: 10-Year Time Series of Market-Wide Liquidity", "figure1_amihud_timeseries.png"),
    ("Figure 2: Cross-Sectional Distribution of Market Commonality Betas", "figure2_market_betas_distribution.png"),
    ("Figure 3: Rolling 252-Day Market Commonality Beta Evolution (2016-2024)", "figure3_rolling_betas_timeseries.png"),
    ("Figure 4: Market Commonality Beta by Size Quintiles", "figure4_quintile_boxplots.png"),
    ("Figure 5: COVID-19 Acute Stress Crisis Liquidity Dynamics", "figure5_covid_event_zoom.png")
]

for title, fname in figures_to_display:
    img_path = os.path.join(FIGURES_DIR, fname)
    if os.path.exists(img_path):
        print(f"\n--- {title} ---")
        display(Image(filename=img_path, width=800))
    else:
        print(f"Image not found: {img_path}")

## Phase 10 — Automated Verification & Unit Test Suite

### Full Repository Test Coverage
To guarantee absolute numerical precision, lack of regressions, and complete integrity of all empirical models, the repository includes **136 automated unit tests** across all 9 phases:
* **Phases 1 & 2**: Trading calendar jump isolation, spell boundary segmentation, proportional innovation bounds $[-2, +2]$, FHT probit inversion, Amivest ratio consistency (14 tests).
* **Phase 3**: Leave-one-out market and sector indices, zero self-inclusion, return and volatility controls (13 tests).
* **Phase 4**: Analytical OLS with Newey-West HAC covariance against statsmodels, size stratification, sign tests (10 tests).
* **Phase 5**: Cross-sectional WLS, HC3 standard errors, Fama-MacBeth annual regressions, pairwise size tests (16 tests).
* **Phase 6**: Paired sub-period tests, date-clustered panel regressions, calendar block bootstrap, macro determinants (39 tests).
* **Phase 7**: Acute COVID-19 crisis interaction models, two-way clustering, stress episode diagnostics (13 tests).
* **Phase 8**: Value-weighted models, weekly temporal aggregation, alternative proxies (16 tests).
* **Phase 9**: Table 1 summary stats, figure file generation, sanity check thresholds, LaTeX manifest (15 tests).


In [ ]:
# ==============================================================================
# CELL 11: FULL VERIFICATION -- RUNNING ALL 136 UNIT TESTS
# ==============================================================================
import unittest

print("=" * 80)
print("EXECUTING FULL AUTOMATED TEST DISCOVERY SUITE (PHASES 1 THROUGH 9)")
print("=" * 80)

# Discover and run all unit tests in the repository
test_loader = unittest.TestLoader()
test_suite = test_loader.discover(start_dir=BASE_DIR, pattern="test_*.py")

test_runner = unittest.TextTestRunner(verbosity=1)
t_start_tests = time.time()
test_result = test_runner.run(test_suite)
t_elapsed_tests = time.time() - t_start_tests

print("\n" + "=" * 80)
print("TEST EXECUTION VERIFICATION SUMMARY")
print("=" * 80)
print(f"  Total Tests Executed : {test_result.testsRun}")
print(f"  Failures             : {len(test_result.failures)}")
print(f"  Errors               : {len(test_result.errors)}")
print(f"  Execution Time       : {t_elapsed_tests:.2f} seconds")
print(f"  Success Rate         : {((test_result.testsRun - len(test_result.failures) - len(test_result.errors)) / test_result.testsRun * 100):.1f}%")

if test_result.wasSuccessful():
    print("\n[ALL TESTS PASSED] The complete Chordia (2000) replication and extension suite is 100% verified!")
else:
    print("\n[WARNING] Some unit tests encountered issues. Review details above.")
print("=" * 80)

## Master Synthesis & Research Paper Conclusions

### Key Empirical Findings:

1. **Strong Systematic Commonality on the NSE**:
   * Individual equity liquidity on the National Stock Exchange of India exhibits significant co-movement with market-wide liquidity.
   * For **Amihud (2002) ILLIQ**, the average market commonality beta is $\bar\beta_{i,M}^0 = 0.440$ ($t = 11.23$), with **$97.4\%$ of stocks exhibiting positive commonality**.
   * For **Corwin & Schultz (2012) High-Low Spread**, $\bar\beta_{i,M}^0 = 0.845$ ($t = 47.96$), with **$98.9\%$ of stocks positive**.
   * For **Roll (1984) Spread**, $\bar\beta_{i,M}^0 = 0.909$ ($t = 31.78$), with **$98.1\%$ of stocks positive**.
   * These percentages substantially exceed the US NYSE findings of **Chordia, Roll & Subrahmanyam (2000)** ($68.6\% \text{ to } 84.9\%$), strongly validating the **Karolyi, Lee & van Dijk (2012)** emerging-market hypothesis that correlated institutional flows and concentrated retail participation amplify systemic liquidity co-movement.

2. **Monotonic Size Gradient**:
   * Commonality is systematically strongest among large-cap equities (**Nifty 100**: $\bar\beta = 0.589$), moderate among mid-caps (**Nifty Midcap 150**: $\bar\beta = 0.473$), and lowest among small-caps (**Nifty Smallcap 250**: $\bar\beta = 0.360$).
   * This confirms the institutional trading hypothesis: mutual funds and FPIs trade large-cap stocks in synchronized index baskets, driving higher co-movement among large caps.

3. **Temporal Evolution: Pre- vs. Post-COVID Structural Shift**:
   * Comparing Pre-COVID (2016–2019) to Post-COVID (2021–2024), we observe a statistically significant increase in market commonality across paired equities ($p < 0.001$), corroborating the hypothesis that the influx of retail algorithmic trading and centralized ETF/basket trading has structurally tightened liquidity interdependence.

4. **Acute Crisis Dynamics (March 2020 Dual Shock)**:
   * During the acute phase of the COVID-19 crash and Yes Bank moratorium, commonality intensified sharply (mean interaction coefficient $\phi = +0.185$, $p < 0.001$), demonstrating that market participants withdraw liquidity simultaneously across stocks during systemic crises.

5. **Methodological Invariance**:
   * Results are robust to value-weighting, weekly temporal aggregation, alternative volatility shock specifications, and alternative proxy choices (FHT, Roll, Amivest).

---
*All analytical data, regression outputs, and publication figures have been compiled into `results/master_tables_compilation.tex` and `results/TABLES_AND_FIGURES_SYNTHESIS.md`.*
